## Configurar el entorno

In [14]:
import os
import boto3
import sagemaker
import pandas as pd
from sagemaker import get_execution_role

# Configuración de sesión y obtención del rol
sess = sagemaker.Session()
role = get_execution_role()

# Identificadores de cuenta y región
account = sess.boto_session.client("sts").get_caller_identity()["Account"]
region = sess.boto_session.region_name

print(f"Cuenta: {account} | Región: {region}")

Cuenta: 150215480648 | Región: us-east-1


## Subir los datos de entrenamiento

In [15]:
WORK_DIRECTORY = "../data/prep" 

# Prefijo y bucket por defecto
prefix = "retail-sales-xgboost-byoc-v3"
default_bucket = sess.default_bucket()

# Carga de datos
data_location = sess.upload_data(WORK_DIRECTORY, bucket=default_bucket, key_prefix=prefix)
print(f"Datos de entrenamiento subidos a: {data_location}")

Datos de entrenamiento subidos a: s3://sagemaker-us-east-1-150215480648/retail-sales-xgboost-byoc-v3


## Crear un estimator y entrenar el modelo

In [16]:
image_name = "sagemaker-xgboost-byoc"
image_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/{image_name}:latest"
s3_output_path = f"s3://{default_bucket}/{prefix}/output"


xgb_estimator = sagemaker.estimator.Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=s3_output_path,
    sagemaker_session=sess,
    hyperparameters={
        "n_estimators": "150",
        "max_depth": "8"
    }
)

xgb_estimator.fit({"training": data_location})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-byoc-2026-03-08-06-17-08-703


2026-03-08 06:17:10 Starting - Starting the training job...
2026-03-08 06:17:23 Starting - Preparing the instances for training...
2026-03-08 06:17:46 Downloading - Downloading input data...
2026-03-08 06:18:22 Training - Training image download completed. Training in progress.Starting the training.
✓ RMSE xgboost (principal): 0.9474
Training complete.

2026-03-08 06:20:10 Uploading - Uploading generated training model
2026-03-08 06:20:10 Completed - Training job completed
Training seconds: 144
Billable seconds: 144


# Hosting del modelo

## Deploy del modelo

In [17]:
from sagemaker.serializers import CSVSerializer

# Desplegamos en una instancia ml.m5.large (como se solicita en la tarea)
predictor = xgb_estimator.deploy(initial_instance_count=1, instance_type="ml.m5.large", serializer=CSVSerializer())

INFO:sagemaker:Creating model with name: sagemaker-xgboost-byoc-2026-03-08-06-24-46-061
INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-byoc-2026-03-08-06-24-46-061
INFO:sagemaker:Creating endpoint with name sagemaker-xgboost-byoc-2026-03-08-06-24-46-061


----!

## Seleccionar datos y usarlos para una predicción

In [18]:
df_val = pd.read_parquet("../data/prep/datos_validacion.parquet")

# Muestra de 5 registros, se elimina "item_cnt_month"
X_sample = df_val.drop(["item_cnt_month"], axis=1).sample(5)

print("Enviando muestra al endpoint...")

# Predicción
response = predictor.predict(X_sample.values).decode("utf-8")

print("\nPredicciones del modelo (unidades estimadas por mes):")
print(response)

Enviando muestra al endpoint...

Predicciones del modelo (unidades estimadas por mes):
0.009431293
0.08580604
0.024992192
0.7194749
0.10781071



## Limpieza opcional

In [ ]:
sess.delete_endpoint(predictor.endpoint)